# Guitar Tuner: Spectral Note Detection and Cent Accuracy

## Objective

- Analyze every WAV file in the project folder or its `data/` subfolder.
- Estimate the fundamental frequency from the spectrum.
- Identify the nearest guitar string and report tuning error in cents.

## Reference data

The string labels follow the assignment notation, from the highest string to the lowest string.

The supplied Hz-per-cent values are retained for the requested resolution calculation.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import signal
from scipy.fft import next_fast_len, rfft, rfftfreq
from scipy.io import wavfile

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

REFERENCE = pd.DataFrame(
    {
        "String": ["E1", "B2", "G3", "D4", "A5", "E6"],
        "Reference frequency (Hz)": [329.63, 246.94, 196.00, 146.83, 110.00, 82.41],
        "Supplied 1-cent step (Hz)": [0.15, 0.15, 0.15, 0.08, 0.08, 0.04],
    }
).sort_values("Reference frequency (Hz)").reset_index(drop=True)

display(REFERENCE.sort_values("Reference frequency (Hz)", ascending=False).reset_index(drop=True))

,String,Reference frequency (Hz),Supplied 1-cent step (Hz)
0,E1,329.63,0.15
1,B2,246.94,0.15
2,G3,196.00,0.15
3,D4,146.83,0.08
4,A5,110.00,0.08
5,E6,82.41,0.04


## Method

- Convert each recording to one floating-point mono signal.
- Remove DC offset and trim low-energy leading and trailing regions.
- Apply a Hann window and compute a zero-padded real FFT.
- Search within 150 cents of each assigned string and sum six weighted harmonics.
- Refine the spectral maximum by parabolic interpolation.
- Select the reference string with the smallest absolute cent distance.

The signed tuning error is $1200\log_2(f_{estimated}/f_{reference})$ cents.

In [2]:
SEARCH_MIN_HZ = 70.0
SEARCH_MAX_HZ = 360.0
DISPLAY_MAX_HZ = 500.0
MAX_ANALYSIS_SECONDS = 8.0
MAX_DETUNING_CENTS = 150.0
ZERO_PADDING_FACTOR = 4
HARMONIC_WEIGHTS = 1.0 / np.sqrt(np.arange(1, 7, dtype=float))


def _to_float_mono(samples):
    """Convert a WAV array to normalized floating-point mono."""
    raw = np.asarray(samples)
    if np.issubdtype(raw.dtype, np.integer):
        info = np.iinfo(raw.dtype)
        if info.min == 0:
            midpoint = (info.max + 1) / 2.0
            audio = (raw.astype(np.float64) - midpoint) / midpoint
        else:
            scale = float(max(abs(info.min), info.max))
            audio = raw.astype(np.float64) / scale
    else:
        audio = raw.astype(np.float64)

    if audio.ndim == 2:
        mixed = audio.mean(axis=1)
        channel_rms = np.sqrt(np.mean(audio**2, axis=0))
        if np.sqrt(np.mean(mixed**2)) < 0.25 * np.max(channel_rms):
            mixed = audio[:, int(np.argmax(channel_rms))]
        audio = mixed
    elif audio.ndim != 1:
        raise ValueError(f"Unsupported WAV shape: {audio.shape}")

    return np.nan_to_num(audio)


def load_wav(path):
    sample_rate, samples = wavfile.read(path)
    audio = _to_float_mono(samples)
    if sample_rate <= 0 or len(audio) < max(32, int(0.10 * sample_rate)):
        raise ValueError("The recording is too short for pitch analysis.")
    return int(sample_rate), audio


def trim_to_active_audio(audio, sample_rate):
    """Trim quiet ends with short-time RMS while preserving the note decay."""
    centered = signal.detrend(audio, type="constant")
    frame = max(32, int(0.050 * sample_rate))
    hop = max(16, frame // 2)
    starts = np.arange(0, max(1, len(centered) - frame + 1), hop)
    rms = np.array([np.sqrt(np.mean(centered[s : s + frame] ** 2)) for s in starts])
    if not len(rms) or np.max(rms) <= np.finfo(float).eps:
        raise ValueError("The recording contains no measurable signal.")

    edge_count = max(1, len(rms) // 10)
    edge_floor = np.median(np.concatenate([rms[:edge_count], rms[-edge_count:]]))
    threshold = min(0.25 * np.max(rms), max(0.04 * np.max(rms), 2.5 * edge_floor))
    active = np.flatnonzero(rms >= threshold)
    if not len(active):
        active = np.array([int(np.argmax(rms))])

    pad = int(0.10 * sample_rate)
    start = max(0, int(starts[active[0]]) - pad)
    stop = min(len(centered), int(starts[active[-1]]) + frame + pad)
    segment = centered[start:stop]
    max_samples = int(MAX_ANALYSIS_SECONDS * sample_rate)
    if len(segment) > max_samples:
        segment = segment[:max_samples]
    if len(segment) < int(0.10 * sample_rate):
        raise ValueError("The active note is too short for pitch analysis.")
    return segment


def calculate_spectrum(audio, sample_rate):
    """Return a normalized one-sided amplitude spectrum."""
    window = signal.windows.hann(len(audio), sym=False)
    windowed = signal.detrend(audio, type="linear") * window
    n_fft = next_fast_len(max(len(audio), ZERO_PADDING_FACTOR * len(audio)))
    spectrum = np.abs(rfft(windowed, n=n_fft))
    frequencies = rfftfreq(n_fft, d=1.0 / sample_rate)
    spectrum /= np.max(spectrum) + np.finfo(float).eps
    return frequencies, spectrum


def _parabolic_frequency(values, index, frequency_step):
    if index <= 0 or index >= len(values) - 1:
        return index * frequency_step
    local = np.log(np.maximum(values[index - 1 : index + 2], np.finfo(float).tiny))
    denominator = local[0] - 2.0 * local[1] + local[2]
    offset = 0.0 if abs(denominator) < 1e-15 else 0.5 * (local[0] - local[2]) / denominator
    return (index + float(np.clip(offset, -1.0, 1.0))) * frequency_step


def estimate_fundamental(frequencies, spectrum):
    """Estimate the fundamental with a weighted harmonic spectrum."""
    range_mask = (frequencies >= SEARCH_MIN_HZ) & (frequencies <= SEARCH_MAX_HZ)
    reference_hz = REFERENCE["Reference frequency (Hz)"].to_numpy()
    distance_to_references = np.abs(1200.0 * np.log2(frequencies[range_mask, None] / reference_hz))
    mask = np.zeros_like(range_mask)
    mask[range_mask] = np.min(distance_to_references, axis=1) <= MAX_DETUNING_CENTS
    candidates = frequencies[mask]
    if not len(candidates):
        raise ValueError("The sample rate is too low for the search range.")

    harmonic_score = np.zeros_like(candidates)
    for harmonic, weight in enumerate(HARMONIC_WEIGHTS, start=1):
        partial_frequency = harmonic * candidates
        partial_amplitude = np.interp(partial_frequency, frequencies, spectrum, left=0.0, right=0.0)
        harmonic_score += weight * partial_amplitude

    local_index = int(np.argmax(harmonic_score))
    frequency_step = frequencies[1] - frequencies[0]
    global_index = np.flatnonzero(mask)[local_index]
    full_score = np.zeros_like(frequencies)
    full_score[mask] = harmonic_score
    estimate_hz = _parabolic_frequency(full_score, global_index, frequency_step)

    baseline = np.median(harmonic_score) + np.finfo(float).eps
    score_ratio_db = 20.0 * np.log10(harmonic_score[local_index] / baseline)
    return float(estimate_hz), float(frequency_step), float(score_ratio_db)


def classify_frequency(estimate_hz):
    refs = REFERENCE["Reference frequency (Hz)"].to_numpy()
    cent_distances = 1200.0 * np.log2(estimate_hz / refs)
    row = REFERENCE.iloc[int(np.argmin(np.abs(cent_distances)))]
    cent_error = 1200.0 * np.log2(estimate_hz / row["Reference frequency (Hz)"])
    return row, float(cent_error)


def analyze_signal(audio, sample_rate, filename="<array>"):
    active = trim_to_active_audio(audio, sample_rate)
    frequencies, spectrum = calculate_spectrum(active, sample_rate)
    estimate_hz, grid_hz, score_ratio_db = estimate_fundamental(frequencies, spectrum)
    reference, cent_error = classify_frequency(estimate_hz)
    direction = "in tune" if abs(cent_error) < 0.5 else ("sharp" if cent_error > 0 else "flat")
    grid_cent = grid_hz / float(reference["Supplied 1-cent step (Hz)"])

    result = {
        "File": str(filename),
        "Detected string": reference["String"],
        "Estimated frequency (Hz)": estimate_hz,
        "Reference frequency (Hz)": float(reference["Reference frequency (Hz)"]),
        "Frequency error (Hz)": estimate_hz - float(reference["Reference frequency (Hz)"]),
        "Tuning error (cent)": cent_error,
        "Absolute error (cent)": abs(cent_error),
        "Direction": direction,
        "FFT grid / supplied cent step": grid_cent,
        "Harmonic peak ratio (dB)": score_ratio_db,
        "Sample rate (Hz)": int(sample_rate),
        "Analyzed duration (s)": len(active) / sample_rate,
    }
    details = {"frequencies": frequencies, "spectrum": spectrum, "estimate_hz": estimate_hz}
    return result, details


def analyze_file(path):
    sample_rate, audio = load_wav(path)
    return analyze_signal(audio, sample_rate, filename=Path(path).name)

## Synthetic validation

Known detunings test note classification and cent estimation before real recordings are analyzed.

In [3]:
def synthesize_guitar_note(
    frequency_hz, sample_rate=44100, duration=4.0, seed=0, fundamental_scale=1.0
):
    rng = np.random.default_rng(seed)
    time = np.arange(int(sample_rate * duration)) / sample_rate
    envelope = np.minimum(time / 0.015, 1.0) * np.exp(-time / 2.8)
    note = np.zeros_like(time)
    for harmonic in range(1, 7):
        amplitude = 0.72 ** (harmonic - 1)
        if harmonic == 1:
            amplitude *= fundamental_scale
        note += amplitude * np.sin(2.0 * np.pi * harmonic * frequency_hz * time)
    note = envelope * note + 0.003 * rng.standard_normal(len(time))
    return 0.8 * note / np.max(np.abs(note))


KNOWN_DETUNING_CENTS = [-7.0, 4.5, -2.5, 8.0, 1.5, -5.0]
validation_rows = []
for seed, (reference, known_cents) in enumerate(
    zip(REFERENCE.itertuples(index=False), KNOWN_DETUNING_CENTS)
):
    reference_hz = reference[1]
    test_hz = reference_hz * 2.0 ** (known_cents / 1200.0)
    test_audio = synthesize_guitar_note(test_hz, seed=seed)
    measured, _ = analyze_signal(test_audio, 44100, filename=f"synthetic_{reference[0]}")
    validation_rows.append(
        {
            "Expected string": reference[0],
            "Detected string": measured["Detected string"],
            "Known detuning (cent)": known_cents,
            "Measured detuning (cent)": measured["Tuning error (cent)"],
            "Estimation difference (cent)": measured["Tuning error (cent)"] - known_cents,
        }
    )

validation = pd.DataFrame(validation_rows)
display(validation.round(3))
assert (validation["Expected string"] == validation["Detected string"]).all()
assert validation["Estimation difference (cent)"].abs().max() < 1.0

weak_audio = synthesize_guitar_note(82.41, duration=1.0, seed=10, fundamental_scale=0.08)
weak_result, _ = analyze_signal(weak_audio, 44100, filename="weak_fundamental_E6")
assert weak_result["Detected string"] == "E6"
print("Synthetic validation passed, including the weak-fundamental test.")

,Expected string,Detected string,Known detuning (cent),Measured detuning (cent),Estimation difference (cent)
0,E6,E6,-7.0,-7.037,-0.037
1,A5,A5,4.5,4.529,0.029
2,D4,D4,-2.5,-2.488,0.012
3,G3,G3,8.0,8.004,0.004
4,B2,B2,1.5,1.495,-0.005
5,E1,E1,-5.0,-4.994,0.006


Synthetic validation passed, including the weak-fundamental test.


## WAV file analysis

Place recordings anywhere below this notebook's folder; `data/` is recommended.

In [4]:
PROJECT_ROOT = Path.cwd()
wav_paths = sorted(
    path for path in PROJECT_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() == ".wav"
)

if wav_paths:
    display(Markdown(f"Found **{len(wav_paths)}** WAV file(s)."))
    for path in wav_paths:
        print(path.relative_to(PROJECT_ROOT))
else:
    display(Markdown("**No WAV files were found; add the recordings and rerun this notebook.**"))

**No WAV files were found; add the recordings and rerun this notebook.**

In [5]:
analysis_rows = []
spectral_details = {}
analysis_errors = []

for path in wav_paths:
    try:
        row, details = analyze_file(path)
        row["File"] = str(path.relative_to(PROJECT_ROOT))
        analysis_rows.append(row)
        spectral_details[str(path.relative_to(PROJECT_ROOT))] = details
    except Exception as exc:
        analysis_errors.append({"File": str(path.relative_to(PROJECT_ROOT)), "Error": str(exc)})

RESULT_COLUMNS = [
    "File",
    "Detected string",
    "Estimated frequency (Hz)",
    "Reference frequency (Hz)",
    "Frequency error (Hz)",
    "Tuning error (cent)",
    "Absolute error (cent)",
    "Direction",
    "FFT grid / supplied cent step",
    "Harmonic peak ratio (dB)",
    "Sample rate (Hz)",
    "Analyzed duration (s)",
]
results = pd.DataFrame(analysis_rows, columns=RESULT_COLUMNS)

if not results.empty:
    display(results.round(3))
if analysis_errors:
    display(Markdown("### Files that could not be analyzed"))
    display(pd.DataFrame(analysis_errors))

In [6]:
if spectral_details:
    figure, axes = plt.subplots(
        len(spectral_details), 1, figsize=(11, 3.2 * len(spectral_details)), squeeze=False
    )
    reference_plot = REFERENCE.sort_values("Reference frequency (Hz)")
    for axis, (filename, details) in zip(axes.ravel(), spectral_details.items()):
        frequencies = details["frequencies"]
        spectrum = details["spectrum"]
        visible = (frequencies >= 60.0) & (frequencies <= DISPLAY_MAX_HZ)
        axis.plot(frequencies[visible], 20.0 * np.log10(spectrum[visible] + 1e-8), lw=1.0)
        for reference in reference_plot.itertuples(index=False):
            axis.axvline(reference[1], color="0.72", lw=0.8, ls=":")
            axis.text(reference[1], -4, reference[0], ha="center", va="top", fontsize=8)
        axis.axvline(details["estimate_hz"], color="#c0392b", lw=1.5, label="Estimated fundamental")
        axis.set(title=filename, xlabel="Frequency (Hz)", ylabel="Relative magnitude (dB)", ylim=(-80, 2))
        axis.legend(loc="lower right")
    figure.tight_layout()
    plt.show()

## Interpretation

- A negative cent value means the played note is flat.
- A positive cent value means the played note is sharp.
- An FFT-grid ratio below 1 is finer than the supplied one-cent frequency step.
- The harmonic peak ratio is a quality indicator, not a statistical confidence interval.
- Very short, noisy, polyphonic, or strongly bent notes can reduce accuracy.

## References

- [Zero Cap Cable frequency reference](http://zerocapcable.com/?page_id=225)
- [Engineering ToolBox octave-band frequency limits](https://www.engineeringtoolbox.com/octave-bands-frequency-limits-d_1602.html)

## Conclusion

The workflow performs reproducible spectral analysis, identifies the nearest assigned guitar string, and reports signed tuning accuracy in cents for every valid WAV file.